[Setup instructions](./Summary.md#topic-notebooks)


In [ ]:
import numpy as np

from pathlib import Path
import sys

_shared_dir = Path.cwd()
if not (_shared_dir / "common.py").is_file():
    _shared_dir = _shared_dir / "02-regression"
if not (_shared_dir / "common.py").is_file():
    raise FileNotFoundError(
        "Run this notebook from the machine-learning or 02-regression folder."
    )
if str(_shared_dir) not in sys.path:
    sys.path.insert(0, str(_shared_dir))

from common import (
    base,
    categories,
    df_train,
    df_val,
    y_train,
    y_val,
    rmse,
)


## Regularization

- Duplicate or linearly dependent features make `XᵀX` **singular**.
- Nearly dependent features can produce unstable, very large weights.
- **Regularization** adds `r` to the diagonal: `XᵀX + rI`.
- This stabilizes training and controls the weights.

### Prepare the features consistently

- Start with a fresh feature list on every call.
- Use the same categories from training data for train and validation.

In [361]:
def prepare_X(df):
    df = df.copy()
    features = base.copy()

    df['age'] = 2017 - df.year
    features.append('age')

    for v in [2, 3, 4]:
        feature = 'num_doors_%s' % v
        df[feature] = (df.number_of_doors == v).astype(int)
        features.append(feature)

    for c, values in categories.items():
        for v in values:
            feature = '%s_%s' % (c, v)
            df[feature] = (df[c] == v).astype(int)
            features.append(feature)

    return df[features].fillna(0).values

### Regularized training function

- `np.eye()` creates the identity matrix.
- `r * np.eye(...)` adds `r` only to the diagonal.
- Like the lesson, this also regularizes the bias term.

In [362]:
def train_linear_regression_reg(X, y, r=0.001):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])

    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

### Evaluate with regularization

In [363]:
X_train = prepare_X(df_train)
w0, w = train_linear_regression_reg(X_train, y_train, r=0.01)
X_val = prepare_X(df_val)
y_pred = w0 + X_val.dot(w)

rmse(y_val, y_pred)

np.float64(0.456521990129513)

- Lesson result with `r=0.01`: RMSE ≈ `0.46` (log-price scale).
- `r=0` → ordinary linear regression.
- Too much regularization can worsen predictions.
- Next: choose `r` using **validation RMSE**.

# Tuning the model

- We already saw that regularization helps stabilize the model when many categorical features are used.
- The regularization parameter `r` controls how strongly we penalize large weights.
- To choose a good value, we try several values and measure the error on the validation set.
- The best value is the one with the lowest validation RMSE.

In [364]:
for r in [0.0, 0.00001, 0.0001, 0.001, 0.1, 1, 10]:
    X_train = prepare_X(df_train)
    w0, w = train_linear_regression_reg(X_train, y_train, r=r)

    X_val = prepare_X(df_val)
    y_pred = w0 + X_val.dot(w)
    score = rmse(y_val, y_pred)

    print(r, w0, score)

0.0 -4385683742668581.5 50.33369424418199
1e-05 8.356193526437128 0.45651704110014846
0.0001 6.2274933923199125 0.4565170627781918
0.001 6.286296964119915 0.45651750887938797
0.1 6.191208737562086 0.45656927630157634
1 5.634896668388998 0.4572204317996368
10 4.283980108973523 0.47014569320977306


What we observe:
- With `r = 0`, the model is unstable and the RMSE is very large.
- A tiny amount of regularization immediately improves the model.
- Too much regularization makes the model underfit and the score gets worse.
- Around `r = 0.001`, the model is already stable and performs well.

This is a classic hyperparameter-tuning step: test a range of values, compare validation errors, and pick the best one.

In [365]:
r = 0.001
X_train = prepare_X(df_train)
w0, w = train_linear_regression_reg(X_train, y_train, r=r)

X_val = prepare_X(df_val)
y_pred = w0 + X_val.dot(w)
score = rmse(y_val, y_pred)
score


np.float64(0.45651750887938797)